In [1]:
import os
import cv2
import joblib
import shap
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, 
                             roc_auc_score, matthews_corrcoef)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, ReLU, LeakyReLU, Add, 
                                     MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Flatten)
from tensorflow.keras.applications import (DenseNet201, VGG16, MobileNetV3Small, 
                                           InceptionV3, ResNet50, ResNet101, DenseNet121)

warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output


In [2]:
train_path = "E:AL/alz/Final_Journal_Simulation/Dataset/ADNI/Final_Dataset/Enhanced_Augmented_6_Class (Different Variations)/AD vs CN vs EMCI vs LMCI/"

In [3]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    height_shift_range=0.2,
    width_shift_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    shear_range=0.2,
    fill_mode='nearest',
    validation_split=0.2
)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [4]:
image_size = (224, 224)
batch = 16
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    class_mode='categorical',
    subset = 'training',
    shuffle=True,
    batch_size=batch
)
test_data = test_datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    class_mode='categorical',
    subset = 'validation',
    shuffle=False,
    batch_size=batch
)

Found 32000 images belonging to 4 classes.
Found 8000 images belonging to 4 classes.


In [15]:
base_model = VGG16(weights='imagenet',include_top=False, input_shape=(224,224,3))
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False
base_model.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_11 (InputLayer)       [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [16]:
model = tf.keras.models.Sequential([
    Input(shape=(224,224,3)),
    base_model,
    GlobalAveragePooling2D(),
    Dense(256,activation='relu'),
    Dropout(0.4),
    Dense(4,activation='softmax')
])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 7, 7, 512)         14714688  
                                                                 
 global_average_pooling2d_1   (None, 512)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dense_2 (Dense)             (None, 256)               131328    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_3 (Dense)             (None, 4)                 1028      
                                                                 
Total params: 14,847,044
Trainable params: 7,211,780
Non-trainable params: 7,635,264
___________________________________

In [5]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, LeakyReLU, MaxPooling2D
from tensorflow.keras.layers import Add, Input, GlobalAveragePooling2D, Dense, Dropout, Concatenate
from tensorflow.keras.layers import Multiply  # For attention
from tensorflow.keras.models import Model

def multi_kernel_conv_block(x, filters, name_prefix):
    """Multi-kernel stem block"""
    x1 = Conv2D(filters//2, (3, 3), padding='same', name=f'{name_prefix}_conv3x3')(x)
    x1 = BatchNormalization(name=f'{name_prefix}_bn3x3')(x1)
    x1 = LeakyReLU(name=f'{name_prefix}_lrelu3x3')(x1)
    
    x2 = Conv2D(filters//2, (5, 5), padding='same', name=f'{name_prefix}_conv5x5')(x)
    x2 = BatchNormalization(name=f'{name_prefix}_bn5x5')(x2)
    x2 = LeakyReLU(name=f'{name_prefix}_lrelu5x5')(x2)
    
    x3 = Conv2D(filters//4, (1, 1), padding='same', name=f'{name_prefix}_conv1x1')(x)
    x3 = BatchNormalization(name=f'{name_prefix}_bn1x1')(x3)
    x3 = LeakyReLU(name=f'{name_prefix}_lrelu1x1')(x3)
    
    x = Concatenate(name=f'{name_prefix}_concat')([x1, x2, x3])
    return x

def attention_block(x, filters, name_prefix):
    """Simple attention mechanism"""
    attention = Conv2D(filters, (1, 1), padding='same', name=f'{name_prefix}_attn_conv')(x)
    attention = BatchNormalization(name=f'{name_prefix}_attn_bn')(attention)
    attention = LeakyReLU(name=f'{name_prefix}_attn_lrelu')(attention)
    attention = Conv2D(filters, (1, 1), padding='same', activation='sigmoid')(attention)
    x = Multiply()([x, attention])
    return x

def residual_block(x, filters, use_projection=True, name_prefix='res'):
    """Residual block with 3x3 kernels"""
    if use_projection:
        shortcut = Conv2D(filters, (1, 1), padding='same', name=f'{name_prefix}_proj')(x)
        shortcut = BatchNormalization(name=f'{name_prefix}_proj_bn')(shortcut)
    else:
        shortcut = x
    
    x = BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = LeakyReLU(name=f'{name_prefix}_lrelu1')(x)
    x = Conv2D(filters, (3, 3), padding='same', name=f'{name_prefix}_conv1')(x)
    
    x = BatchNormalization(name=f'{name_prefix}_bn2')(x)
    x = LeakyReLU(name=f'{name_prefix}_lrelu2')(x)
    x = Conv2D(filters, (3, 3), padding='same', name=f'{name_prefix}_conv2')(x)
    
    x = Add(name=f'{name_prefix}_add')([x, shortcut])
    return x

def create_focus_resnet_model(input_shape=(224, 224, 3), num_classes=4):
    # Input Layer
    inputs = Input(shape=input_shape, name='input_layer')

    # Stem with multi-kernel
    x = multi_kernel_conv_block(inputs, 32, 'stem')
    x = MaxPooling2D((2, 2), name='stem_pool')(x)

    # Stage 1
    x = residual_block(x, 64, name_prefix='stage1_res1')
    x = attention_block(x, 64, 'stage1_attn')
    x = MaxPooling2D((2, 2), name='stage1_pool')(x)

    # Stage 2
    x = residual_block(x, 128, name_prefix='stage2_res1')
    x = residual_block(x, 128, name_prefix='stage2_res2')
    x = attention_block(x, 128, 'stage2_attn')
    x = MaxPooling2D((2, 2), name='stage2_pool')(x)

    # Stage 3
    x = residual_block(x, 256, name_prefix='stage3_res1')
    x = residual_block(x, 256, name_prefix='stage3_res2')
    x = attention_block(x, 256, 'stage3_attn')
    x = MaxPooling2D((2, 2), name='stage3_pool')(x)

    # # Stage 4
    # x = residual_block(x, 256, name_prefix='stage4_res1')
    # x = residual_block(x, 256, name_prefix='stage4_res2')
    # x = attention_block(x, 256, 'stage4_attn')
    # x = MaxPooling2D((2, 2), name='stage4_pool')(x)
    
    # Stage 5
    x = residual_block(x, 512, name_prefix='stage4_res1')
    x = attention_block(x, 512, 'stage4_attn')

    # Global pooling and classification
    x = GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = Dense(256, name='fc1',kernel_regularizer=regularizers.l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU()(x)
    x = Dropout(0.4)(x)
    
    x = Dense(128, name='fc2',kernel_regularizer=regularizers.l2(0.00001))(x)
    x = BatchNormalization()(x)
    x = LeakyReLU()(x)
    x = Dropout(0.4)(x)

    # Output
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    # Create model
    model = Model(inputs, outputs, name='focus_resnet_model')
    return model

# Example usage
# model = create_medical_model(input_shape=(224, 224, 3), num_classes=4)
# model.summary()

In [14]:
model= create_focus_resnet_model(input_shape=(224, 224, 3), num_classes=4)
model.summary()

Model: "focus_resnet_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_layer (InputLayer)       [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 stem_conv3x3 (Conv2D)          (None, 224, 224, 16  448         ['input_layer[0][0]']            
                                )                                                                 
                                                                                                  
 stem_conv5x5 (Conv2D)          (None, 224, 224, 16  1216        ['input_layer[0][0]']            
                                )                                                

In [7]:
early_stopping = EarlyStopping(
        patience = 5, 
        monitor = 'val_loss',
        restore_best_weights=True,
        mode='min'
)
lr_reduce = ReduceLROnPlateau(
        patience=2,
        factor=0.5,
        min_lr=1e-6,
        monitor='val_loss'
)

In [8]:
import numpy as np

# Assuming `test_generator` is your ImageDataGenerator
X_test = []
y_test = []

# Loop through the generator to extract all images and labels
for images, labels in test_data:
    X_test.append(images)
    y_test.append(labels)
   
    # Stop when we have gone through the entire dataset
    if len(X_test) * test_data.batch_size >= test_data.samples:
        break

# Convert to NumPy arrays
X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_test shape: (8000, 224, 224, 3)
y_test shape: (8000, 4)


In [9]:
def calc_multiclass_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]  # Number of classes in confusion matrix
    expected_classes = 4  # We expect 4 classes
    classes = np.arange(expected_classes)  # [0, 1, 2, 3]
    
    sensitivities = []
    specificities = []
    for i in range(expected_classes):
        if i < n_classes:  # If class exists in confusion matrix
            tp = cm[i, i]
            fn = cm[i, :].sum() - tp
            fp = cm[:, i].sum() - tp
            tn = cm.sum() - (tp + fp + fn)
        else:  # If class is missing
            tp = fn = fp = tn = 0
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        sensitivities.append(sensitivity)
        specificities.append(specificity)
    
    macro_sensitivity = np.mean(sensitivities)
    macro_specificity = np.mean(specificities)
    
    if n_classes < expected_classes:
        print(f"Warning: Only {n_classes} classes detected in confusion matrix, expected {expected_classes}")
    
    return sensitivities, specificities, macro_sensitivity, macro_specificity

In [15]:
model.compile(optimizer=Adam(learning_rate=0.001,beta_2=.999, beta_1=0.5),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

In [ ]:
history = model.fit(train_data, epochs=50, validation_data=test_data, callbacks=[lr_reduce,early_stopping],verbose=2)

Epoch 1/50
